# 2 · From a MIDI file to a new melody

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sem-16161d/GreasyPidginTutorial/blob/main/EN/jupyter/markov-book-en/02_from_midi_to_melody.ipynb)

Now the symbols are **notes**. The plan:

1. choose a Scarlatti sonata (default: K.555)
2. let GreasyPidgin read it
3. learn which note tends to follow which
4. generate a new melody
5. write it to a MIDI file you can download

In [ ]:
# Setup -- runs by itself (about 20 seconds the first time)
import importlib.util, pathlib, subprocess, sys, urllib.request

GP_REF        = "main"   # which version of GreasyPidgin to use (a git branch or tag)
TUTORIAL_REF  = "main"   # which version of this tutorial's helper file to use
GP_REPO       = "sem-16161d/GreasyPidgin"
TUTORIAL_REPO = "sem-16161d/GreasyPidginTutorial"

if importlib.util.find_spec("GreasyPidgin") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check",
                    f"git+https://github.com/{GP_REPO}.git@{GP_REF}"], check=True)

if not pathlib.Path("workshop_helpers.py").exists():
    urllib.request.urlretrieve(
        f"https://raw.githubusercontent.com/{TUTORIAL_REPO}/{TUTORIAL_REF}/EN/jupyter/markov-book-en/workshop_helpers.py",
        "workshop_helpers.py")

import random
import matplotlib.pyplot as plt
from GreasyPidgin import MarkovChain, MarkovRule, MarkovRules
import workshop_helpers as wh

print("Ready!")

## Step 1: choose a sonata

The training material is one of Domenico Scarlatti's 555 keyboard sonatas (Kirkpatrick numbering, K.1 to K.555).
The default is the last one: **K.555**.

**Change the number in the cell below** (1 to 555) and run the cell to train on a different sonata.
The sonatas have two hands; if several notes start together, only the highest one is kept.
A MIDI file of your own? See "Try it" at the very bottom. (MIDI files: © Sankey, according to the file names.)

In [ ]:
SONATA = 555      # <- change this number (1 to 555) to train on a different sonata

midi_path = wh.get_scarlatti(SONATA, TUTORIAL_REPO, TUTORIAL_REF)

## Step 2: read it with GreasyPidgin

Behind the scenes, `Gesture.fromMidiFile` turns every MIDI note into a **Phonon** —
GreasyPidgin's note object — and collects them in a **Gesture**.
We take pitches and rhythms out of it and keep them in a small `Melody` object.

In [ ]:
melody = wh.load_melody(midi_path)
print(melody)
wh.plot_melody(melody)
plt.show()

In [ ]:
wh.listen(midi_path)

`melody.pitches` are MIDI note numbers (60 = middle C, written `c4`; `s` means sharp and `f` flat, so `cs4` is C#4 and `bf3` is Bb3), `melody.deltas` are the times between
note starts in beats (1.0 = one quarter note), `melody.bpm` is the tempo.

In [ ]:
print(melody.pitches[:12])
print(melody.deltas[:12])
print(melody.bpm)

## Step 3: learn the chain

`ORDER` is the memory: how many previous notes decide the next one.

In [ ]:
ORDER = 2

pitch_chain = MarkovChain()
pitch_chain.generateRulesFromList(melody.pitches, order=ORDER)
print(f"{len(pitch_chain.rules)} rules learned. The first three:")

for state in pitch_chain.rules.elements[:3]:
    names = " ".join(wh.pitch_name(p) for p in state)
    print(f"  after  {names:8s} ->", pitch_chain.rules[state])

## Step 4: generate

We start the new melody with the first notes of the original and let the chain continue.

In [ ]:
N_NOTES = 32

start = tuple(melody.pitches[:ORDER])
new_pitches = pitch_chain.getNElements(seed=start, n=N_NOTES)

print(" ".join(wh.pitch_name(p) for p in new_pitches))

## Step 5: a rhythm, and a MIDI file

For now the new melody borrows the rhythm of the original (repeating it as often as needed).
The next chapter teaches the chain rhythm, too.

We build a **Gesture** from parallel lists: one list for pitches, one for the time between
starts, one for durations, plus a dynamic and a tempo.

In [ ]:
from GreasyPidgin import Gesture

rhythm = [melody.deltas[i % len(melody.deltas)] for i in range(len(new_pitches))]

gesture = Gesture.fromLists(
    size        = len(new_pitches),
    startOffset = 0,
    startDeltas = [0] + rhythm[:-1],           # time before each note starts
    durations   = [r * 0.9 for r in rhythm],   # slightly detached
    pitches     = new_pitches,
    dynamics    = 80,                          # MIDI velocity
    timeUnit    = "b",                         # all times are in beats
    bpm         = melody.bpm,
)

gesture.toMidi("new_melody.mid", bpm=melody.bpm)

In [ ]:
wh.plot_notes(new_pitches, rhythm, title="The new melody", color="C1")
plt.show()
wh.listen("new_melody.mid")

Happy with it? Save the MIDI file to your computer:

In [ ]:
wh.download("new_melody.mid")

## A different approach: intervals instead of notes

So far the chain has learned **absolute notes**: "after `c5` comes `ef5`". It only plays notes that occurred in the
original, always in the same register. Now let it learn **intervals** instead: the distance between two consecutive
notes in semitones (`+2` = a major second up, `-3` = a minor third down).
An interval like "two semitones up" occurs in many places, whatever the note. So the chain learns the *shape* of the
melody, not its pitches.

In [ ]:
intervals = [b - a for a, b in zip(melody.pitches, melody.pitches[1:])]
print(intervals[:12])      # semitone steps between consecutive notes

interval_chain = MarkovChain()
interval_chain.generateRulesFromList(intervals, order=ORDER)
print(f"Learned {len(interval_chain.rules)} rules (with absolute notes there were {len(pitch_chain.rules)}).")

To generate, we start on the first note of the original and add the learned intervals one after another.
If the melody leaves the range of the original, we move it back by octaves.

In [ ]:
low, high = min(melody.pitches), max(melody.pitches)

new_intervals = interval_chain.getNElements(seed=tuple(intervals[:ORDER]), n=N_NOTES)

pitch = melody.pitches[0]            # start note: the first note of the original
interval_pitches = [pitch]
for step in new_intervals[:N_NOTES - 1]:
    pitch += step
    while pitch > high:              # stay within the range of the original
        pitch -= 12
    while pitch < low:
        pitch += 12
    interval_pitches.append(pitch)

print(" ".join(wh.pitch_name(p) for p in interval_pitches))

In [ ]:
wh.render_midi(interval_pitches, rhythm, melody.bpm, "new_melody_intervals.mid")
wh.plot_notes(interval_pitches, rhythm, title="The new melody (intervals)", color="C2")
plt.show()
wh.listen("new_melody_intervals.mid")

Listen to both melodies one after the other. The note chain stays with the tones and turns of the original. The interval
chain puts its movements together anew at arbitrary places, and can hit tones that never occurred in the original.

In [ ]:
wh.download("new_melody_intervals.mid")

## Again, again!

Go back to **Step 4** and run the cells below it again: every run is a new melody.

**Try it:**

- Change `ORDER` to `1` or `3` in Step 3 (and run Step 3 and 4 again). Does the new melody still
  sound like the original?
- Make it longer (`N_NOTES = 100`).
- Start the interval melody on a different note: set `pitch = 60` instead of `pitch = melody.pitches[0]`.
  It has the same shape, just shifted.
- Pick a different sonata: change `SONATA` in Step 1 and run all cells from Step 1 again.
- Upload a MIDI file of your own: replace the `midi_path = ...` line in Step 1 with `midi_path = wh.choose_midi()`.
- Change `0.9` in Step 5 to `1.0` (legato) or `0.4` (staccato).